# PopGenLM Bench v0.3 · evolutionary-evidence results

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahirali-biomics/popgenlm-bench/blob/v0.3.0/notebooks/popgenlm-bench-v0.3.ipynb)

This lightweight publication notebook validates and summarizes the frozen v0.3 release assets. It prefers local repository files and otherwise uses the pinned `v0.3.0` raw GitHub tree. It performs no model inference, VCF processing, UMAP fitting, bootstrap resampling, or other production analysis.

The primary population-oriented model score is `log P(minor allele) - log P(major allele)`: negative values favor the major allele term, while positive values favor the minor allele term. Raw REF-to-ALT scores are kept separately as `log P(ALT) - log P(REF)`. PhyloP is a signed site-oriented conservation score, not an allele-oriented score.

All reported correlations and bootstrap intervals are descriptive; no hypothesis-test p-values are claimed. The notebook display and summaries are distinct from the precomputed production results.

## 1. Locate and verify the release

Every local or downloaded asset is checked against `manifest.json` before it is parsed. The manifest itself is also verified against its expected release SHA-256.

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
from IPython.display import Image, Markdown, display

RELEASE_REF = "v0.3.0"
RAW_BASE = "https://raw.githubusercontent.com/tahirali-biomics/popgenlm-bench/v0.3.0/"
MANIFEST_REL = "data/benchmarks/v0.3/manifest.json"
ROOT = next((candidate for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent) if (candidate / MANIFEST_REL).exists()), None)

def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def fetch(relative_path: str) -> bytes:
    local = ROOT / relative_path if ROOT is not None else None
    if local is not None and local.exists():
        return local.read_bytes()
    with urlopen(RAW_BASE + relative_path) as response:
        return response.read()

manifest_bytes = fetch(MANIFEST_REL)
manifest = json.loads(manifest_bytes)
assert manifest["release"] == RELEASE_REF
assets = {asset["path"]: asset for asset in manifest["assets"]}
assert len(assets) == 14

def verified(relative_path: str) -> bytes:
    data = fetch(relative_path)
    expected = assets[relative_path]
    assert sha256(data) == expected["sha256"], f"SHA-256 mismatch: {relative_path}"
    assert len(data) == expected["bytes"], f"Byte-size mismatch: {relative_path}"
    return data

print("Using local repository files" if ROOT is not None else f"Using {RAW_BASE}")
print(f"Verified manifest for {len(assets)} promoted assets")

## 2. Load and validate all released tables

The tables below are precomputed production results. In particular, the annotated master table must have exactly 10,000 rows and 39 columns.

In [ ]:
tsv_assets = [path for path in assets if path.endswith(".tsv")]
tables = {}
for path in tsv_assets:
    asset = assets[path]
    table = pd.read_csv(io.BytesIO(verified(path)), sep="\t")
    assert table.shape == (asset["rows"], asset["columns"]), path
    tables[path] = table

master_path = "data/benchmarks/v0.3/popgenlm_v03_master_10000_annotated.tsv"
master = tables[master_path]
assert master.shape == (10000, 39), "The annotated master table must have 10,000 rows and 39 columns"
required_master = {"chrom", "pos", "ref", "alt", "gpn_score_ref_alt", "gpn_score_minor_vs_major", "plantcad_score_ref_alt", "plantcad_score_minor_vs_major", "phylop", "primary_context"}
assert required_master <= set(master.columns)
print(f"Validated {len(tables)} TSV tables; annotated master: {master.shape[0]:,} rows × {master.shape[1]} columns")
pd.DataFrame([{"path": path, "rows": table.shape[0], "columns": table.shape[1]} for path, table in tables.items()])

## 3. Benchmark and annotation overview

The overview is a display of released annotations and population-oriented score columns; it does not calculate new production scores.

In [ ]:
quality = tables["data/benchmarks/v0.3/embeddings/sample_quality.tsv"]
contexts = master["primary_context"].value_counts().rename_axis("primary_context").reset_index(name="variants")
overview = pd.Series({
    "variants": len(master),
    "chromosomes": master["chrom"].nunique(),
    "samples in quality table": len(quality),
    "primary samples": int(quality["primary_include"].eq(True).sum()),
    "strict samples": int(quality["strict_include"].eq(True).sum()),
    "samples with geographic metadata": int(quality["geographic_metadata_available"].eq(True).sum()),
})
display(overview.to_frame("value"))
display(contexts)

## 4. Model concordance and LD sensitivity

The following values are selected directly from the released statistics tables. The population-oriented GPN–PlantCAD Spearman correlation is 0.6525 overall, with physical-bin bootstrap intervals supplied by production. LD profiles show the descriptive estimate is stable across the released thinning settings.

In [ ]:
overall = tables["data/benchmarks/v0.3/statistics/overall_ld_sensitivity_correlations.tsv"]
bootstrap = tables["data/benchmarks/v0.3/statistics/physical_bin_bootstrap_ci.tsv"]
concordance = overall[overall["comparison"].eq("gpn_vs_plantcad_minor_major") & overall["method"].eq("spearman")].copy()
concordance["estimate"] = concordance["estimate"].astype(float)
display(concordance[["scope", "n", "estimate"]].reset_index(drop=True))
selected_ci = bootstrap[(bootstrap["comparison"] == "gpn_vs_plantcad_minor_major") & (bootstrap["method"] == "spearman") & (bootstrap["bin_width_bp"] == 100000)]
display(selected_ci[["bin_width_bp", "estimate", "ci_lower_2.5", "ci_upper_97.5", "bootstrap_replicates"]].reset_index(drop=True))
print("Interpretation: these are descriptive correlations and bootstrap intervals; no hypothesis-test p-values are claimed.")

## 5. Genotype UMAP sample-quality summary

The UMAP coordinates and sample-quality classifications are loaded from the release. The notebook does not fit or rotate a new embedding.

In [ ]:
umap_summary = pd.DataFrame({
    "analysis_set": ["primary", "strict"],
    "samples": [int(quality["primary_include"].eq(True).sum()), int(quality["strict_include"].eq(True).sum())],
    "call_rate_threshold": [">= 0.80", ">= 0.90"],
})
display(umap_summary)
print(f"Mean call rate in quality table: {quality['call_rate'].mean():.3f}")

## 6. Released figures

These are the two precomputed publication figures. PNGs are displayed here; SVG versions are linked for vector use.

In [ ]:
for png, svg, title in [(
    "figures/v0.3/model_evidence.png", "figures/v0.3/model_evidence.svg", "Model evidence",
), (
    "figures/v0.3/genotype_umap_geography.png", "figures/v0.3/genotype_umap_geography.svg", "Genotype UMAP and geography",
)]:
    display(Markdown(f"### {title} · [`SVG`]({RAW_BASE + svg})"))
    display(Image(data=verified(png)))